## 2.7 Vector Search with `sqlitesearch` (Part 2)

In [3]:
import torch
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


Creating embedding model and vector search index:

In [5]:
model = SentenceTransformer("all-MiniLM-L6-v2", device=device)

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="fac_vectors2.db",
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [8]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vs_index.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5,
)

In [ ]:
results

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'doc_id': '74eb249bbf'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.',
  'doc_id': '69d122f12e'},
 {'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the course be offere

In [10]:
from rag_helper import RAGBase

class RAGVector(RAGBase):
    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query: str, num_results: int = 5) -> str:
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict,
        )

In [11]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

The RAG now uses `sqlitesearch`:

In [12]:
vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=openai_client,
)

In [13]:
vector_assistant.rag('the program has already begun, can I still sign up?')

'Yes, you can still join. You can start learning and submitting homework while the form is open, even if the course has already begun. If you want a certificate, make sure to submit your project before submissions close.'

In [14]:
vector_assistant.rag('whats queen gambit?')

"I don't know."